# Inference Latency and Batching

Week 3-4 starts with a production question: once a model exists, how do we serve it fast and predictably?

This notebook builds the benchmarking instincts you need before deploying to Triton, TorchServe, BentoML, or any other serving stack. We will measure local PyTorch inference, compare batch sizes, inspect latency percentiles, and reason about throughput vs tail latency.

## What you will learn

By the end, you should be comfortable with these serving concepts:

- **Latency**: how long one request takes.
- **Throughput**: how many examples or requests are processed per second.
- **Batching**: grouping multiple examples into one model forward pass.
- **Warmup**: ignoring first calls that include setup/cache effects.
- **p50/p95/p99**: latency percentiles, especially tail latency.
- **Optimization baseline**: a measurement you can compare against after changing something.

## 0. Setup

This notebook uses PyTorch for the local model and pandas/matplotlib for reporting. It runs on CPU, but if CUDA is available the benchmark will use it.

In [ ]:
import math
import statistics
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as exc:
    raise ImportError("PyTorch is required. Install it with: pip install torch") from exc

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. A small model to serve

Serving systems do not care whether the model is glamorous. The same ideas apply to classifiers, recommenders, embedding models, and LLMs.

We will use a small MLP with a realistic input/output shape. The model is not trained for accuracy here; it exists so we can study inference behavior.

In [ ]:
class TinyFraudScorer(nn.Module):
    def __init__(self, n_features=64, hidden=256, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)

model = TinyFraudScorer().to(device).eval()
num_params = sum(p.numel() for p in model.parameters())
example = torch.randn(1, 64, device=device)
with torch.inference_mode():
    logits = model(example)
    probabilities = F.softmax(logits, dim=-1)

print(model)
print(f"Parameters: {num_params:,}")
print(f"Example probabilities: {probabilities.detach().cpu().numpy().round(4)}")

## 2. Correct timing basics

A quick `time.time()` around one forward pass is usually misleading. Better inference timing includes:

- `model.eval()` to disable training behavior.
- `torch.inference_mode()` to skip autograd overhead.
- warmup iterations before measurement.
- many repeated samples.
- synchronization when measuring CUDA kernels.

The helper below handles these basics.

In [ ]:
def sync_if_needed():
    if device == "cuda":
        torch.cuda.synchronize()

def percentile(values, q):
    if not values:
        return float("nan")
    sorted_values = sorted(values)
    idx = min(len(sorted_values) - 1, max(0, math.ceil(q / 100 * len(sorted_values)) - 1))
    return sorted_values[idx]

@dataclass
class BenchmarkResult:
    batch_size: int
    examples_per_call: int
    calls: int
    mean_ms: float
    p50_ms: float
    p95_ms: float
    p99_ms: float
    throughput_examples_per_sec: float

def benchmark_forward(model, batch_size, n_features=64, warmup=25, repeats=150):
    x = torch.randn(batch_size, n_features, device=device)
    model.eval()

    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(x)
        sync_if_needed()

        latencies_ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            _ = model(x)
            sync_if_needed()
            latencies_ms.append((time.perf_counter() - start) * 1000)

    mean_ms = statistics.mean(latencies_ms)
    total_examples = batch_size * repeats
    total_seconds = sum(latencies_ms) / 1000
    throughput = total_examples / total_seconds

    return BenchmarkResult(
        batch_size=batch_size,
        examples_per_call=batch_size,
        calls=repeats,
        mean_ms=mean_ms,
        p50_ms=percentile(latencies_ms, 50),
        p95_ms=percentile(latencies_ms, 95),
        p99_ms=percentile(latencies_ms, 99),
        throughput_examples_per_sec=throughput,
    )

## 3. Single-request baseline

The first baseline is batch size 1. This approximates a synchronous service that handles one request at a time.

The latency number matters, but the percentile shape matters more. Production systems often optimize for p95 or p99, not just average latency.

In [ ]:
baseline = benchmark_forward(model, batch_size=1)
pd.DataFrame([baseline.__dict__])

## 4. Batch-size sweep

Batching usually improves throughput because one forward pass can process many examples with better hardware utilization. But batching can increase per-request wait time if requests have to queue until a batch fills.

The benchmark below only measures model execution time, not queueing delay. Triton dynamic batching later adds the queueing side of the story.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
results = []

for batch_size in batch_sizes:
    result = benchmark_forward(model, batch_size=batch_size)
    results.append(result.__dict__)

df = pd.DataFrame(results)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df["batch_size"], df["p50_ms"], marker="o", label="p50")
axes[0].plot(df["batch_size"], df["p95_ms"], marker="o", label="p95")
axes[0].plot(df["batch_size"], df["p99_ms"], marker="o", label="p99")
axes[0].set_xscale("log", base=2)
axes[0].set_xlabel("batch size")
axes[0].set_ylabel("latency per forward call (ms)")
axes[0].set_title("Latency percentiles")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(df["batch_size"], df["throughput_examples_per_sec"], marker="o", color="tab:green")
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("batch size")
axes[1].set_ylabel("examples/sec")
axes[1].set_title("Throughput")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Per-example latency vs per-call latency

A batch of 64 may take longer than a batch of 1, but less than 64 separate calls. That is the throughput win.

For online APIs, you care about request latency. For offline batch scoring, you often care more about examples per second.

In [ ]:
df = df.assign(
    mean_ms_per_example=df["mean_ms"] / df["batch_size"],
    p95_ms_per_example=df["p95_ms"] / df["batch_size"],
)
df[["batch_size", "mean_ms", "mean_ms_per_example", "p95_ms", "p95_ms_per_example", "throughput_examples_per_sec"]]

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(df["batch_size"], df["mean_ms_per_example"], marker="o")
plt.xscale("log", base=2)
plt.xlabel("batch size")
plt.ylabel("mean ms per example")
plt.title("Batching reduces execution cost per example")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Compare normal mode, no_grad, and inference_mode

During training, PyTorch tracks gradients. During inference, that is wasted work.

- Normal mode: builds autograd graph.
- `torch.no_grad()`: disables gradient tracking.
- `torch.inference_mode()`: disables gradient tracking and extra version counter work.

For production inference, `inference_mode` is usually the best default.

In [ ]:
def time_mode(mode_name, batch_size=32, repeats=150, warmup=25):
    x = torch.randn(batch_size, 64, device=device)
    latencies = []

    def run_once():
        if mode_name == "normal":
            return model(x)
        if mode_name == "no_grad":
            with torch.no_grad():
                return model(x)
        if mode_name == "inference_mode":
            with torch.inference_mode():
                return model(x)
        raise ValueError(mode_name)

    for _ in range(warmup):
        _ = run_once()
    sync_if_needed()

    for _ in range(repeats):
        start = time.perf_counter()
        _ = run_once()
        sync_if_needed()
        latencies.append((time.perf_counter() - start) * 1000)

    return {
        "mode": mode_name,
        "mean_ms": statistics.mean(latencies),
        "p95_ms": percentile(latencies, 95),
    }

mode_df = pd.DataFrame([time_mode(name) for name in ["normal", "no_grad", "inference_mode"]])
mode_df

## 7. Optional: TorchScript export

Triton can serve several model formats. One path is TorchScript. The main idea is to turn Python model code into a serialized artifact that can be loaded by a serving runtime.

This is a local export example, not yet a Triton deployment.

In [ ]:
import os

artifact_dir = "artifacts"
os.makedirs(artifact_dir, exist_ok=True)

scripted = torch.jit.trace(model.cpu(), torch.randn(1, 64))
scripted_path = os.path.join(artifact_dir, "tiny_fraud_scorer.pt")
scripted.save(scripted_path)
model.to(device)

loaded = torch.jit.load(scripted_path).eval()
with torch.inference_mode():
    test_out = loaded(torch.randn(4, 64))

print(f"Saved TorchScript model to {scripted_path}")
print(f"Loaded model output shape: {tuple(test_out.shape)}")

## 8. Production interpretation

A serving benchmark is only useful if it maps to a product requirement.

Example requirements:

- Fraud score API: p95 under 50 ms at 200 requests/sec.
- Offline nightly scoring: process 50 million rows in under 2 hours.
- Embedding service: p99 under 250 ms while keeping GPU utilization above 60%.

These goals imply different batching and scaling choices.

In [ ]:
best_throughput = df.loc[df["throughput_examples_per_sec"].idxmax()]
lowest_p95 = df.loc[df["p95_ms"].idxmin()]

print("Best throughput setting:")
print(best_throughput[["batch_size", "throughput_examples_per_sec", "p95_ms"]])
print("\nLowest p95 latency setting:")
print(lowest_p95[["batch_size", "throughput_examples_per_sec", "p95_ms"]])

## 9. Exercises

1. Increase the hidden size from 256 to 1024. How do latency and throughput change?
2. Increase the number of repeated layers. Does p95 grow linearly?
3. Try batch sizes beyond 128. Where does throughput stop improving?
4. Export the model with ONNX if `onnx` and `onnxruntime` are installed.
5. Write a short paragraph explaining the batch size you would choose for an online fraud scoring API.

## Summary

You now have the baseline skill for Week 3-4: measuring inference performance.

Key takeaways:

- Always warm up before timing.
- Use inference mode for PyTorch serving.
- Track p50, p95, and p99, not only mean latency.
- Batching improves throughput, but service queueing can affect tail latency.
- A benchmark table is the evidence you need before optimizing or deploying.

Next notebook: build the Triton model repository and client flow.